# Vnořené dotazy, LIKE, IN, BETWEEN

V předchozím notebooku jsme se naučili filtrovat záznamy pomocí `WHERE` a logických operátorů. Nyní rozšíříme možnosti filtrování o **pokročilé operátory** a **vnořené dotazy** (subqueries).

---

## Přehled operátorů v tomto notebooku

| Operátor | Popis | Příklad |
|---|---|---|
| `IN` | Hodnota se nachází **v seznamu** | `WHERE znacka IN ('Skoda', 'Audi')` |
| `NOT IN` | Hodnota **není v seznamu** | `WHERE znacka NOT IN ('Skoda')` |
| `BETWEEN` | Hodnota je **v rozsahu** (včetně hranic) | `WHERE cena BETWEEN 100 AND 500` |
| `NOT BETWEEN` | Hodnota **není v rozsahu** | `WHERE cena NOT BETWEEN 100 AND 500` |
| `LIKE` | Text **odpovídá vzoru** (pattern matching) | `WHERE jmeno LIKE 'J%'` |
| `NOT LIKE` | Text **neodpovídá vzoru** | `WHERE jmeno NOT LIKE '%ova'` |
| Vnořený dotaz | `SELECT` uvnitř jiného `SELECT` | `WHERE cena > (SELECT AVG(cena) ...)` |

Všechny tyto operátory se používají v klauzuli `WHERE`.

## Instalace balíčku

In [ ]:
# Instalace knihovny (stačí spustit jednou)
! pip install mysql-connector-python

## Připojení k databázi

Přihlašovací údaje importujeme z modulu `pripojeni.py`.

In [ ]:
import mysql.connector
from pripojeni import *  # importuje konstanty HOST, USER, PASSWORD, DATABASE

mydb = mysql.connector.connect(
    host=HOST,
    user=USER,
    password=PASSWORD,
    database=DATABASE
)

mycursor = mydb.cursor()

## Příprava ukázkových dat

Pro všechny příklady si vytvoříme tabulku `automobily` se sloupcem `znacka`:

In [ ]:
mycursor.execute("DROP TABLE IF EXISTS automobily")

mycursor.execute("""
    CREATE TABLE automobily (
        id INT PRIMARY KEY AUTO_INCREMENT,
        spz CHAR(7) NOT NULL,
        znacka VARCHAR(30) NOT NULL,
        pocet_sedadel INT NOT NULL,
        max_rychlost INT,
        nosnost INT,
        nutna_kvalifikace VARCHAR(10) NOT NULL DEFAULT 'B',
        datum_vyroby DATE
    )
""")

mycursor.execute("""
    INSERT INTO automobily (spz, znacka, pocet_sedadel, max_rychlost, nosnost, nutna_kvalifikace, datum_vyroby) VALUES
        ('1A11111', 'Skoda',   4, 150, 3, 'B', '2000-09-09'),
        ('2B22222', 'Ford',    2, 220, 2, 'B', '2020-01-01'),
        ('3B33333', 'Audi',    4, 220, 2, 'B', '2018-01-01'),
        ('4B44444', 'Skoda',   4, 190, 4, 'B', '2017-05-11'),
        ('5A55555', 'Ferrari', 2, 250, 3, 'B', '2005-09-09')
""")

mydb.commit()
print("✅ Tabulka 'automobily' vytvořena a naplněna (5 záznamů).")

---

# IN — výběr z množiny hodnot

Operátor `IN` slouží k filtrování řádků, jejichž hodnota se nachází **v zadaném seznamu**. Je to zkrácený zápis pro více podmínek spojených operátorem `OR`.

## Syntaxe

```sql
SELECT sloupce FROM tabulka
WHERE sloupec IN (hodnota1, hodnota2, ...);
```

| Zápis s IN | Ekvivalent s OR |
|---|---|
| `WHERE znacka IN ('Skoda', 'Audi')` | `WHERE znacka = 'Skoda' OR znacka = 'Audi'` |
| `WHERE id IN (1, 3, 5)` | `WHERE id = 1 OR id = 3 OR id = 5` |

> **Tip:** `IN` je přehlednější než řetězení `OR`, zvlášť při více hodnotách.

### Příklad

In [ ]:
# Vypíšeme auta značky Skoda a Audi
mycursor.execute("""
    SELECT id, spz, znacka, max_rychlost
    FROM automobily
    WHERE znacka IN ('Skoda', 'Audi')
""")

print("Auta značky Skoda nebo Audi:")
for id, spz, znacka, rychlost in mycursor.fetchall():
    print(f"  Auto #{id}: SPZ={spz}, značka={znacka}, rychlost={rychlost} km/h")

### NOT IN — vyloučení z množiny

In [ ]:
# Auta, která NEJSOU značky Skoda ani Ford
mycursor.execute("""
    SELECT id, spz, znacka
    FROM automobily
    WHERE znacka NOT IN ('Skoda', 'Ford')
""")

print("Auta mimo Skoda a Ford:")
for id, spz, znacka in mycursor.fetchall():
    print(f"  Auto #{id}: SPZ={spz}, značka={znacka}")

---

# LIKE — vyhledávání podle vzoru

Operátor `LIKE` umožňuje filtrovat **textové hodnoty** podle vzoru (pattern). Používá se společně s `WHERE`.

## Syntaxe

```sql
SELECT sloupce FROM tabulka
WHERE sloupec LIKE 'vzor';
```

## Zástupné znaky (wildcards)

| Znak | Popis | Příklad | Najde |
|---|---|---|---|
| `%` | Nahrazuje **0 a více** libovolných znaků | `'F%'` | Ford, Ferrari, Fiat |
| `_` | Nahrazuje **přesně 1** libovolný znak | `'F_r_'` | Ford, ale NE Ferrari |

## Běžné vzory

| Vzor | Význam |
|---|---|
| `'F%'` | Začíná na **F** |
| `'%da'` | Končí na **da** |
| `'%err%'` | Obsahuje **err** (kdekoliv) |
| `'_o%'` | Druhý znak je **o** |
| `'A___'` | Začíná na **A** a má přesně 4 znaky |

> **⚠️ Pozor:** `LIKE` v MySQL je ve výchozím nastavení **case-insensitive** (nerozlišuje velká/malá písmena).

### Příklad — vzor s `%`

In [ ]:
# Značky začínající na 'F'
mycursor.execute("""
    SELECT id, znacka
    FROM automobily
    WHERE znacka LIKE 'F%'
""")

print("Značky začínající na F:")
for id, znacka in mycursor.fetchall():
    print(f"  Auto #{id}: značka={znacka}")

In [ ]:
# Značky obsahující písmeno 'o' (kdekoliv)
mycursor.execute("""
    SELECT id, znacka
    FROM automobily
    WHERE znacka LIKE '%o%'
""")

print("Značky obsahující 'o':")
for id, znacka in mycursor.fetchall():
    print(f"  Auto #{id}: značka={znacka}")

### Příklad — vzor s `_`

In [ ]:
# Značky odpovídající vzoru: F + libovolný znak + r + libovolný znak  →  "F_r_"
mycursor.execute("""
    SELECT id, znacka
    FROM automobily
    WHERE znacka LIKE 'F_r_'
""")

print("Značky odpovídající vzoru 'F_r_':")
for id, znacka in mycursor.fetchall():
    print(f"  Auto #{id}: značka={znacka}")

# ☝️ Ford (F-o-r-d) odpovídá, Ferrari NE (má více znaků)

---

# BETWEEN — rozsah hodnot

Operátor `BETWEEN` vybírá řádky, jejichž hodnota leží **v zadaném rozsahu** (včetně obou hranic). Funguje pro čísla, text i datum.

## Syntaxe

```sql
SELECT sloupce FROM tabulka
WHERE sloupec BETWEEN hodnota1 AND hodnota2;
```

| Zápis s BETWEEN | Ekvivalent |
|---|---|
| `WHERE max_rychlost BETWEEN 160 AND 230` | `WHERE max_rychlost >= 160 AND max_rychlost <= 230` |
| `WHERE datum_vyroby BETWEEN '2010-01-01' AND '2020-12-31'` | `WHERE datum_vyroby >= '2010-01-01' AND datum_vyroby <= '2020-12-31'` |

> **Tip:** `BETWEEN` je **inkluzivní** — zahrnuje obě hraniční hodnoty.

### Příklad — číselný rozsah

In [ ]:
# Auta s maximální rychlostí 160–230 km/h
mycursor.execute("""
    SELECT id, znacka, max_rychlost
    FROM automobily
    WHERE max_rychlost BETWEEN 160 AND 230
""")

print("Auta s rychlostí 160–230 km/h:")
for id, znacka, rychlost in mycursor.fetchall():
    print(f"  Auto #{id}: značka={znacka}, rychlost={rychlost} km/h")

### Příklad — datumový rozsah

In [ ]:
# Auta vyrobená mezi roky 2010 a 2020
mycursor.execute("""
    SELECT id, znacka, datum_vyroby
    FROM automobily
    WHERE datum_vyroby BETWEEN '2010-01-01' AND '2020-12-31'
""")

print("Auta vyrobená 2010–2020:")
for id, znacka, datum in mycursor.fetchall():
    print(f"  Auto #{id}: značka={znacka}, vyrobeno={datum}")

### Příklad — textový rozsah (abecední)

In [ ]:
# Značky abecedně mezi 'Audi' a 'Ford' (včetně)
mycursor.execute("""
    SELECT id, znacka, spz
    FROM automobily
    WHERE znacka BETWEEN 'Audi' AND 'Ford'
""")

print("Značky abecedně Audi–Ford:")
for id, znacka, spz in mycursor.fetchall():
    print(f"  Auto #{id}: značka={znacka}, SPZ={spz}")

---

# Vnořené dotazy (subqueries)

Vnořený dotaz je `SELECT` **uvnitř jiného** SQL příkazu. Vnitřní dotaz se vyhodnotí **jako první** a jeho výsledek se použije ve vnějším dotazu.

## Syntaxe

```sql
SELECT sloupce FROM tabulka
WHERE sloupec operátor (SELECT ... FROM ...);
```

## Kdy je použít?

Vnořené dotazy se hodí, když potřebujete **nejdříve vypočítat hodnotu** (průměr, maximum, …) a pak ji použít v podmínce:

| Úloha | SQL |
|---|---|
| Auta rychlejší než průměr | `WHERE max_rychlost > (SELECT AVG(max_rychlost) FROM automobily)` |
| Auta s nejvyšší nosností | `WHERE nosnost = (SELECT MAX(nosnost) FROM automobily)` |
| Auta stejné značky jako id=1 | `WHERE znacka = (SELECT znacka FROM automobily WHERE id = 1)` |

> **Tip:** Vnořený dotaz vracející **jednu hodnotu** se používá s operátory `=`, `>`, `<` atd. Dotaz vracející **více hodnot** se používá s `IN`.

### Příklad — porovnání s průměrem

In [ ]:
# Auta s rychlostí vyšší než průměr
mycursor.execute("""
    SELECT id, znacka, max_rychlost
    FROM automobily
    WHERE max_rychlost > (SELECT AVG(max_rychlost) FROM automobily)
""")

print("Auta rychlejší než průměr:")
for id, znacka, rychlost in mycursor.fetchall():
    print(f"  Auto #{id}: značka={znacka}, rychlost={rychlost} km/h")

### Příklad — vnořený dotaz s další podmínkou

In [ ]:
# Auta značky Audi, která jsou rychlejší než průměr
mycursor.execute("""
    SELECT id, znacka, max_rychlost
    FROM automobily
    WHERE max_rychlost > (SELECT AVG(max_rychlost) FROM automobily)
      AND znacka = 'Audi'
""")

print("Audi rychlejší než průměr:")
for id, znacka, rychlost in mycursor.fetchall():
    print(f"  Auto #{id}: značka={znacka}, rychlost={rychlost} km/h")

### Příklad — vnořený dotaz s IN

In [ ]:
# Auta, jejichž značka se vyskytuje víckrát (skupiny s COUNT > 1)
# → vnořený dotaz vrací více hodnot → použijeme IN
mycursor.execute("""
    SELECT id, znacka, spz
    FROM automobily
    WHERE znacka IN (
        SELECT znacka
        FROM automobily
        GROUP BY znacka
        HAVING COUNT(*) > 1
    )
""")

print("Auta značek, které se vyskytují víckrát:")
for id, znacka, spz in mycursor.fetchall():
    print(f"  Auto #{id}: značka={znacka}, SPZ={spz}")

## Odpojení a úklid

In [ ]:
mycursor.execute("DROP TABLE IF EXISTS automobily")
mydb.commit()

mycursor.close()
mydb.close()

print("✅ Odpojení proběhlo úspěšně.")

---

# Cvičení

Zde bude následovat série úkolů. V každém cvičení se nejprve připojte k databázi pomocí konstant z modulu `pripojeni`. Na konci se vždy odpojte.

> Tabulka `automobily` je v každém cvičení předvytvořena — neupravujte přípravný kód.

## Cvičení 1 — IN

Z tabulky `automobily` vypište záznamy, které mají `datum_vyroby` rovno `'2018-01-01'` **nebo** `'2005-09-09'`.

Použijte operátor `IN`.

<details>
<summary>🔎 Očekávaný výstup (klikni pro zobrazení)</summary>

```
Auto #3: SPZ=3B33333, značka=Audi, sedadel=4, rychlost=220, vyrobeno=2018-01-01
Auto #5: SPZ=5A55555, značka=Ferrari, sedadel=2, rychlost=250, vyrobeno=2005-09-09
```

</details>

In [ ]:
import mysql.connector
from pripojeni import *

mydb = mysql.connector.connect(
    host=HOST, user=USER, password=PASSWORD, database=DATABASE
)
mycursor = mydb.cursor()

# --- tuto část neměnit! (příprava dat) ---
mycursor.execute("DROP TABLE IF EXISTS automobily")
mycursor.execute("""
    CREATE TABLE automobily (
        id INT PRIMARY KEY AUTO_INCREMENT,
        spz CHAR(7) NOT NULL,
        znacka VARCHAR(30) NOT NULL,
        pocet_sedadel INT NOT NULL,
        max_rychlost INT,
        nosnost INT,
        nutna_kvalifikace VARCHAR(10) NOT NULL DEFAULT 'B',
        datum_vyroby DATE
    )
""")
mycursor.execute("""
    INSERT INTO automobily (spz, znacka, pocet_sedadel, max_rychlost, nosnost, nutna_kvalifikace, datum_vyroby) VALUES
        ('1A11111', 'Skoda',   4, 150, 3, 'B', '2000-09-09'),
        ('2B22222', 'Ford',    2, 220, 2, 'B', '2020-01-01'),
        ('3B33333', 'Audi',    4, 220, 2, 'B', '2018-01-01'),
        ('4B44444', 'Skoda',   4, 190, 4, 'B', '2017-05-11'),
        ('5A55555', 'Ferrari', 2, 250, 3, 'B', '2005-09-09')
""")
mydb.commit()

# TODO: Vypište záznamy s datum_vyroby '2018-01-01' nebo '2005-09-09' pomocí IN →


# --- tuto část neměnit! (úklid) ---
mycursor.execute("DROP TABLE automobily")
mydb.commit()
mycursor.close()
mydb.close()

## Cvičení 2 — LIKE

Z tabulky `automobily` vypište záznamy, které mají v atributu `spz` písmeno **B**.

Použijte operátor `LIKE`.

<details>
<summary>🔎 Očekávaný výstup (klikni pro zobrazení)</summary>

```
Auto #2: SPZ=2B22222, značka=Ford
Auto #3: SPZ=3B33333, značka=Audi
Auto #4: SPZ=4B44444, značka=Skoda
```

</details>

In [ ]:
import mysql.connector
from pripojeni import *

mydb = mysql.connector.connect(
    host=HOST, user=USER, password=PASSWORD, database=DATABASE
)
mycursor = mydb.cursor()

# --- tuto část neměnit! (příprava dat) ---
mycursor.execute("DROP TABLE IF EXISTS automobily")
mycursor.execute("""
    CREATE TABLE automobily (
        id INT PRIMARY KEY AUTO_INCREMENT,
        spz CHAR(7) NOT NULL,
        znacka VARCHAR(30) NOT NULL,
        pocet_sedadel INT NOT NULL,
        max_rychlost INT,
        nosnost INT,
        nutna_kvalifikace VARCHAR(10) NOT NULL DEFAULT 'B',
        datum_vyroby DATE
    )
""")
mycursor.execute("""
    INSERT INTO automobily (spz, znacka, pocet_sedadel, max_rychlost, nosnost, nutna_kvalifikace, datum_vyroby) VALUES
        ('1A11111', 'Skoda',   4, 150, 3, 'B', '2000-09-09'),
        ('2B22222', 'Ford',    2, 220, 2, 'B', '2020-01-01'),
        ('3B33333', 'Audi',    4, 220, 2, 'B', '2018-01-01'),
        ('4B44444', 'Skoda',   4, 190, 4, 'B', '2017-05-11'),
        ('5A55555', 'Ferrari', 2, 250, 3, 'B', '2005-09-09')
""")
mydb.commit()

# TODO: Vypište záznamy, které mají v SPZ písmeno B →


# --- tuto část neměnit! (úklid) ---
mycursor.execute("DROP TABLE automobily")
mydb.commit()
mycursor.close()
mydb.close()

## Cvičení 3 — BETWEEN

Z tabulky `automobily` vypište záznamy, jejichž `znacka` je abecedně **mezi** `'Audi'` a `'Ford'` (včetně).

Použijte operátor `BETWEEN`.

<details>
<summary>🔎 Očekávaný výstup (klikni pro zobrazení)</summary>

```
Auto #2: SPZ=2B22222, značka=Ford
Auto #3: SPZ=3B33333, značka=Audi
Auto #5: SPZ=5A55555, značka=Ferrari
```

</details>

In [ ]:
import mysql.connector
from pripojeni import *

mydb = mysql.connector.connect(
    host=HOST, user=USER, password=PASSWORD, database=DATABASE
)
mycursor = mydb.cursor()

# --- tuto část neměnit! (příprava dat) ---
mycursor.execute("DROP TABLE IF EXISTS automobily")
mycursor.execute("""
    CREATE TABLE automobily (
        id INT PRIMARY KEY AUTO_INCREMENT,
        spz CHAR(7) NOT NULL,
        znacka VARCHAR(30) NOT NULL,
        pocet_sedadel INT NOT NULL,
        max_rychlost INT,
        nosnost INT,
        nutna_kvalifikace VARCHAR(10) NOT NULL DEFAULT 'B',
        datum_vyroby DATE
    )
""")
mycursor.execute("""
    INSERT INTO automobily (spz, znacka, pocet_sedadel, max_rychlost, nosnost, nutna_kvalifikace, datum_vyroby) VALUES
        ('1A11111', 'Skoda',   4, 150, 3, 'B', '2000-09-09'),
        ('2B22222', 'Ford',    2, 220, 2, 'B', '2020-01-01'),
        ('3B33333', 'Audi',    4, 220, 2, 'B', '2018-01-01'),
        ('4B44444', 'Skoda',   4, 190, 4, 'B', '2017-05-11'),
        ('5A55555', 'Ferrari', 2, 250, 3, 'B', '2005-09-09')
""")
mydb.commit()

# TODO: Vypište záznamy se značkou abecedně mezi Audi a Ford →


# --- tuto část neměnit! (úklid) ---
mycursor.execute("DROP TABLE automobily")
mydb.commit()
mycursor.close()
mydb.close()

## Cvičení 4 — Vnořený dotaz

Z tabulky `automobily` vypište záznamy, které mají `nosnost` **menší než průměrná nosnost**, nebo `znacka = 'Ferrari'`.

Použijte **vnořený dotaz** pro výpočet průměru a operátor `OR` pro druhou podmínku.

<details>
<summary>🔎 Očekávaný výstup (klikni pro zobrazení)</summary>

```
Auto #2: SPZ=2B22222, značka=Ford, nosnost=2
Auto #3: SPZ=3B33333, značka=Audi, nosnost=2
Auto #5: SPZ=5A55555, značka=Ferrari, nosnost=3
```

</details>

In [ ]:
import mysql.connector
from pripojeni import *

mydb = mysql.connector.connect(
    host=HOST, user=USER, password=PASSWORD, database=DATABASE
)
mycursor = mydb.cursor()

# --- tuto část neměnit! (příprava dat) ---
mycursor.execute("DROP TABLE IF EXISTS automobily")
mycursor.execute("""
    CREATE TABLE automobily (
        id INT PRIMARY KEY AUTO_INCREMENT,
        spz CHAR(7) NOT NULL,
        znacka VARCHAR(30) NOT NULL,
        pocet_sedadel INT NOT NULL,
        max_rychlost INT,
        nosnost INT,
        nutna_kvalifikace VARCHAR(10) NOT NULL DEFAULT 'B',
        datum_vyroby DATE
    )
""")
mycursor.execute("""
    INSERT INTO automobily (spz, znacka, pocet_sedadel, max_rychlost, nosnost, nutna_kvalifikace, datum_vyroby) VALUES
        ('1A11111', 'Skoda',   4, 150, 3, 'B', '2000-09-09'),
        ('2B22222', 'Ford',    2, 220, 2, 'B', '2020-01-01'),
        ('3B33333', 'Audi',    4, 220, 2, 'B', '2018-01-01'),
        ('4B44444', 'Skoda',   4, 190, 4, 'B', '2017-05-11'),
        ('5A55555', 'Ferrari', 2, 250, 3, 'B', '2005-09-09')
""")
mydb.commit()

# TODO: Vypište auta s nosností < průměr NEBO značka = 'Ferrari' →


# --- tuto část neměnit! (úklid) ---
mycursor.execute("DROP TABLE automobily")
mydb.commit()
mycursor.close()
mydb.close()

## Cvičení 5 — Kompletní úloha

Celý úkol je na vás od začátku do konce:

1. Připojte se k databázi.
2. Vytvořte tabulku `knihy` s atributy:
   - `id` — INT, PRIMARY KEY, AUTO_INCREMENT
   - `nazev` — VARCHAR(100), NOT NULL
   - `autor` — VARCHAR(50), NOT NULL
   - `zanr` — VARCHAR(30), NOT NULL
   - `pocet_stran` — INT, NOT NULL
   - `rok_vydani` — INT
3. Vložte **alespoň 8 knih** od **minimálně 3 různých autorů** a ve **3 různých žánrech**.
4. Vypište:
   - Knihy žánru `'fantasy'` nebo `'sci-fi'` — pomocí `IN`
   - Knihy, jejichž název **začíná** na `'H'` — pomocí `LIKE`
   - Knihy s počtem stran **200–400** — pomocí `BETWEEN`
   - Knihy s více stranami, než je **průměr** — pomocí **vnořeného dotazu**
   - Autory, kteří napsali knihu s více stranami než průměr — vnořený dotaz + `IN`
5. Smažte tabulku a odpojte se.

<details>
<summary>🔎 Očekávaný výstup (klikni pro zobrazení)</summary>

```
--- Žánr fantasy nebo sci-fi (IN) ---
...

--- Název začínající na H (LIKE) ---
...

--- 200–400 stran (BETWEEN) ---
...

--- Více stran než průměr (vnořený dotaz) ---
...

--- Autoři s nadprůměrnou knihou (vnořený + IN) ---
...
```

</details>

In [ ]:
import mysql.connector
from pripojeni import *

mydb = mysql.connector.connect(
    host=HOST, user=USER, password=PASSWORD, database=DATABASE
)
mycursor = mydb.cursor()

# TODO: Celý úkol je na vás →


# Nezapomeňte na konci smazat tabulku a odpojit se!